# Prerequisites

In [1]:
!pip install -q requests beautifulsoup4 pandas lxml feedparser tqdm googlenewsdecoder

import re
import os
import base64
import time
import random
import hashlib
import warnings
from datetime import datetime, timedelta
from urllib.parse import quote_plus, urlparse

import requests
import feedparser
import pandas as pd
from bs4 import BeautifulSoup, NavigableString

try:
    from tqdm.notebook import tqdm
except ImportError:
    from tqdm import tqdm

# googlenewsdecoder
try:
    from googlenewsdecoder import new_decoderv1, new_decoderv2
    HAS_DECODER_V1 = True
    HAS_DECODER_V2 = True
    print("✅ googlenewsdecoder v1 + v2 loaded.")
except ImportError:
    try:
        from googlenewsdecoder import new_decoderv1
        HAS_DECODER_V1 = True
        HAS_DECODER_V2 = False
        print("✅ googlenewsdecoder v1 loaded (v2 not available).")
    except ImportError:
        HAS_DECODER_V1 = False
        HAS_DECODER_V2 = False
        print("⚠️ googlenewsdecoder not available, using fallback.")

warnings.filterwarnings("ignore")
print("✅ All libraries loaded.")

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.5/81.5 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 31.6 MB/s eta 0:00:00
✅ googlenewsdecoder v1 + v2 loaded.
✅ All libraries loaded.


# Topic Init

In [2]:
TOPIC_VARIANTS = [
    # ── Inti isu (frasa exact) ────────────────────────
    '"Trump tariff"',
    '"tarif Trump"',
    '"perang dagang"',
    '"trade war"',
    '"reciprocal tariff"',
    '"tarif resiprokal"',

    # ── Target negara & kawasan ──────────────────────
    'Trump tariff Indonesia',
    'Trump tariff ASEAN',
    'Trump tariff China',
    'Trump tariff Asia',
    'tarif Trump Indonesia',
    'dampak tarif Trump Indonesia',
    'perang dagang AS China',
    'perang dagang Amerika Indonesia',

    # ── Kebijakan & regulasi ─────────────────────────
    'Trump trade policy 2025',
    'kebijakan tarif Amerika Serikat',
    'tarif impor Amerika',
    'bea masuk Trump',
    'proteksionisme Trump',
    'tarif barang China Amerika',
    'sanksi dagang Trump',
    'trade deal Amerika Indonesia',
    'negosiasi dagang Indonesia AS',
    'perjanjian dagang Indonesia Amerika',

    # ── Dampak ekonomi Indonesia ─────────────────────
    'ekspor Indonesia terdampak tarif',
    'industri Indonesia tarif Trump',
    'tekstil Indonesia tarif Amerika',
    'sawit Indonesia tarif Trump',
    'tarif Trump UMKM Indonesia',
    'investasi asing tarif Trump',
    'rupiah perang dagang',
    'IHSG perang dagang',
    'ekspor impor Indonesia Amerika',
    'neraca dagang Indonesia AS',

    # ── Dampak global ────────────────────────────────
    'global trade war 2025',
    'WTO Trump tariff',
    'IMF perang dagang',
    'resesi global tarif Trump',
    'supply chain tarif Trump',
    'harga barang naik tarif Trump',
    'inflasi Amerika tarif',
    'ekonomi global terdampak tarif',

    # ── Reaksi pemerintah & pelaku usaha ────────────
    'respons Indonesia tarif Trump',
    'Prabowo tarif Trump',
    'Airlangga tarif Amerika',
    'Sri Mulyani perang dagang',
    'KADIN tarif Trump',
    'pengusaha Indonesia tarif Trump',
    'asosiasi ekspor tarif Amerika',
    'DPR Indonesia tarif Trump',

    # ── Pro kontra & opini ───────────────────────────
    'pro kontra tarif Trump',
    'kritik tarif Trump',
    'dampak positif tarif Trump',
    'dampak negatif tarif Trump',
    'ekonom tarif Trump',
    'analis tarif Amerika',
    'opini perang dagang',

    # ── Sektor spesifik ──────────────────────────────
    'nikel Indonesia tarif Trump',
    'baja aluminium tarif Trump',
    'teknologi tarif Trump',
    'otomotif tarif Trump',
    'pertanian tarif Trump',
    'minyak sawit tarif Amerika',
    'elektronik tarif Trump',
    'sepatu tekstil tarif Amerika',

    # ── English queries ──────────────────────────────
    'Trump tariff impact Southeast Asia',
    'Trump trade war Indonesia 2025',
    'US tariff Indonesia export',
    'Trump reciprocal tariff ASEAN 2025',
    'Indonesia US trade deal 2025',
]

LANGUAGE = "id"
COUNTRY = "ID"
MAX_ARTICLES = 1500

DATE_FROM = "2025-01-20"    # Trump mulai menjabat kembali
DATE_TO = "2026-03-09"      # sampai hari ini
DATE_CHUNK_DAYS = 5

REQUEST_TIMEOUT = 20
DELAY_MIN = 0.5
DELAY_MAX = 1.2

OUTPUT_CSV = "news_articles_trump_tariff.csv"
CHECKPOINT_CSV = "news_checkpoint_trump_tariff.csv"
CHECKPOINT_EVERY = 50

print(f"🔎 Keywords  : {len(TOPIC_VARIANTS)} variasi")
print(f"📅 Rentang   : {DATE_FROM} → {DATE_TO}")
print(f"🎯 Target    : {MAX_ARTICLES} artikel")

🔎 Keywords  : 70 variasi
📅 Rentang   : 2025-01-20 → 2026-03-09
🎯 Target    : 1500 artikel


# URL Scrapping

In [3]:
USER_AGENTS = [
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/125.0.0.0 Safari/537.36",
    "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/125.0.0.0 Safari/537.36",
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:126.0) Gecko/20100101 Firefox/126.0",
    "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/605.1.15 (KHTML, like Gecko) Version/17.4 Safari/605.1.15",
    "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/125.0.0.0 Safari/537.36",
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36",
    "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36",
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:125.0) Gecko/20100101 Firefox/125.0",
    "Mozilla/5.0 (X11; Ubuntu; Linux x86_64; rv:126.0) Gecko/20100101 Firefox/126.0",
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/123.0.0.0 Safari/537.36 Edg/123.0.0.0",
]


def _make_session(ua=None):
    s = requests.Session()
    s.headers.update({
        "User-Agent": ua or random.choice(USER_AGENTS),
        "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8",
        "Accept-Language": "en-US,en;q=0.9,id;q=0.8",
        "Accept-Encoding": "gzip, deflate, br",
        "DNT": "1",
        "Connection": "keep-alive",
        "Upgrade-Insecure-Requests": "1",
        # Referer dari Google — situs lebih menerima traffic dari search engine
        "Referer": "https://www.google.com/",
    })
    adapter = requests.adapters.HTTPAdapter(
        max_retries=requests.adapters.Retry(
            total=3, backoff_factor=1,
            status_forcelist=[429, 500, 502, 503, 504],
        )
    )
    s.mount("http://", adapter)
    s.mount("https://", adapter)
    return s


SESSION = _make_session()


def _rotate_session():
    """Buat session baru dengan UA berbeda."""
    global SESSION
    SESSION = _make_session()


def _delay():
    time.sleep(random.uniform(DELAY_MIN, DELAY_MAX))


def _url_hash(url):
    clean = re.sub(r"[?#].*$", "", url).rstrip("/").lower()
    return hashlib.md5(clean.encode()).hexdigest()


GOOGLE_DOMAINS = {
    "google.com", "google.co.id", "googleapis.com",
    "googleusercontent.com", "gstatic.com", "google.co",
    "googlesyndication.com", "googleadservices.com",
    "youtube.com", "ggpht.com", "feedburner.com",
}


def _is_google_domain(url):
    domain = urlparse(url).netloc.lower()
    return any(g in domain for g in GOOGLE_DOMAINS)


print("✅ Session ready (10 UA rotation).")

✅ Session ready (10 UA rotation).


In [4]:
def decode_google_news_url(google_url: str) -> str:
    """
    Decode Google News URL → URL asli.
    Chain: decoderv1 → decoderv2 → manual base64 → HTTP redirect
    """
    if "news.google.com" not in google_url:
        return google_url

    # ── 1. googlenewsdecoder v1 (fastest, base64-based) ───
    if HAS_DECODER_V1:
        try:
            result = new_decoderv1(google_url, interval=0.3)
            if result and result.get("decoded_url"):
                url = result["decoded_url"]
                if not _is_google_domain(url) and _is_valid_article_url(url):
                    return url
        except Exception:
            pass

    # ── 2. googlenewsdecoder v2 (HTTP-based, lebih reliable) ──
    if HAS_DECODER_V2:
        try:
            result = new_decoderv2(google_url, interval=0.5)
            if result and result.get("decoded_url"):
                url = result["decoded_url"]
                if not _is_google_domain(url) and _is_valid_article_url(url):
                    return url
        except Exception:
            pass

    # ── 3. Enhanced base64 decode ─────────────────────────
    url = _decode_base64_enhanced(google_url)
    if url:
        return url

    # ── 4. HTTP redirect follow ───────────────────────────
    url = _follow_redirects(google_url)
    if url:
        return url

    return google_url


def _is_valid_article_url(url):
    """Validasi URL adalah artikel berita yang valid."""
    try:
        parsed = urlparse(url)
        # Harus punya domain
        if not parsed.netloc or len(parsed.netloc) < 4:
            return False
        # Bukan file media
        path_lower = parsed.path.lower()
        media_ext = ('.jpg', '.jpeg', '.png', '.gif', '.webp', '.svg',
                     '.mp4', '.mp3', '.pdf', '.zip', '.ico')
        if any(path_lower.endswith(ext) for ext in media_ext):
            return False
        return True
    except Exception:
        return False


def _decode_base64_enhanced(google_url):
    """Enhanced base64 decoder: coba multiple parsing strategies."""
    try:
        match = re.search(r"/articles/([A-Za-z0-9_-]+)", google_url)
        if not match:
            return None

        encoded = match.group(1)

        # Coba multiple padding
        decoded_bytes = None
        for padding in ["", "=", "==", "==="]:
            try:
                decoded_bytes = base64.urlsafe_b64decode(encoded + padding)
                break
            except Exception:
                continue
        if not decoded_bytes:
            return None

        decoded_str = decoded_bytes.decode("latin-1", errors="ignore")

        # Cari semua URL
        all_urls = re.findall(
            r"https?://[a-zA-Z0-9._~:/?#\[\]@!$&'()*+,;=%-]+",
            decoded_str,
        )
        if not all_urls:
            return None

        # Filter: buang Google URLs + media files
        candidates = [
            u for u in all_urls
            if not _is_google_domain(u) and _is_valid_article_url(u)
        ]

        if candidates:
            # Prefer URL terpanjang (biasanya lebih spesifik = artikel)
            best = max(candidates, key=len)
            best = re.split(r'[\x00-\x1f\x7f-\x9f]', best)[0]
            best = best.rstrip('.,;:!?)]\'"')
            return best

    except Exception:
        pass
    return None


def _follow_redirects(google_url):
    """Follow HTTP redirects + parse JS redirects."""
    try:
        resp = SESSION.get(
            google_url, allow_redirects=True,
            timeout=REQUEST_TIMEOUT, stream=True,
        )
        if not _is_google_domain(resp.url):
            return resp.url

        content = resp.content[:15000].decode("utf-8", errors="ignore")

        # Parse various redirect patterns
        patterns = [
            r'<meta[^>]*?url=(https?://[^"\'>\s]+)',
            r'data-url="(https?://[^"]+)"',
            r'href="(https?://[^"]+)"',
            r"window\.location\s*=\s*['\"]?(https?://[^'\";\s]+)",
            r'"(https?://[^"]+)"',  # catch-all for JSON-like URLs
        ]
        for pat in patterns:
            for m in re.finditer(pat, content, re.I):
                url = m.group(1)
                if not _is_google_domain(url) and _is_valid_article_url(url):
                    return url

    except Exception:
        pass
    return None


print("✅ Enhanced Google News decoder ready.")
print(f"   Strategies: ", end="")
strats = []
if HAS_DECODER_V1: strats.append("v1")
if HAS_DECODER_V2: strats.append("v2")
strats.extend(["base64", "HTTP"])
print(" → ".join(strats))

✅ Enhanced Google News decoder ready.
   Strategies: v1 → v2 → base64 → HTTP


In [5]:
def generate_date_ranges(start, end, chunk_days):
    fmt = "%Y-%m-%d"
    d_start = datetime.strptime(start, fmt)
    d_end = datetime.strptime(end, fmt)
    ranges = []
    cursor = d_start
    while cursor < d_end:
        chunk_end = min(cursor + timedelta(days=chunk_days - 1), d_end)
        ranges.append((cursor.strftime(fmt), chunk_end.strftime(fmt)))
        cursor = chunk_end + timedelta(days=1)
    return ranges


def fetch_rss(topic, after, before, lang, country):
    query_str = f"{topic} after:{after} before:{before}"
    url = (
        f"https://news.google.com/rss/search"
        f"?q={quote_plus(query_str)}&hl={lang}&gl={country}&ceid={country}:{lang}"
    )
    try:
        feed = feedparser.parse(url)
    except Exception:
        return []

    articles = []
    for entry in feed.entries:
        try:
            dt = datetime(*entry.published_parsed[:6])
            date_str = dt.strftime("%Y-%m-%d %H:%M")
        except Exception:
            date_str = entry.get("published", "")
        articles.append({
            "title": entry.get("title", "").strip(),
            "link": entry.get("link", "").strip(),
            "date": date_str,
        })
    return articles


def collect_all_links():
    date_ranges = generate_date_ranges(DATE_FROM, DATE_TO, DATE_CHUNK_DAYS)
    total_queries = len(TOPIC_VARIANTS) * len(date_ranges)
    print(f"📡 {len(TOPIC_VARIANTS)} keywords × {len(date_ranges)} chunks = {total_queries} queries")

    seen = set()
    articles = []
    pbar = tqdm(total=total_queries, desc="📡 Fetching RSS", unit="query")

    for topic in TOPIC_VARIANTS:
        for (d_after, d_before) in date_ranges:
            if len(articles) >= MAX_ARTICLES:
                pbar.close()
                return articles
            results = fetch_rss(topic, d_after, d_before, LANGUAGE, COUNTRY)
            for art in results:
                h = _url_hash(art["link"])
                if h not in seen:
                    seen.add(h)
                    articles.append(art)
                    if len(articles) >= MAX_ARTICLES:
                        break
            pbar.update(1)
            _delay()

    pbar.close()
    return articles


articles_list = collect_all_links()
print(f"\n✅ {len(articles_list):,} unique links terkumpul.")

📡 70 keywords × 83 chunks = 5810 queries


📡 Fetching RSS:   0%|          | 0/5810 [00:00<?, ?query/s]


✅ 1,500 unique links terkumpul.


In [6]:
print(f"🔗 Decoding {len(articles_list)} URLs...\n")

decode_ok = 0
decode_fail = 0

pbar = tqdm(total=len(articles_list), desc="🔗 Decoding", unit="url")

for art in articles_list:
    original = art["link"]
    decoded = decode_google_news_url(original)
    art["link"] = decoded

    if not _is_google_domain(decoded):
        decode_ok += 1
    else:
        decode_fail += 1

    pbar.update(1)

pbar.close()

# Deduplicate
seen = set()
deduped = []
for art in articles_list:
    h = _url_hash(art["link"])
    if h not in seen:
        seen.add(h)
        deduped.append(art)
articles_list = deduped

print(f"\n   ✅ Decoded : {decode_ok:,}")
print(f"   ❌ Failed  : {decode_fail:,}")
print(f"   📄 Unique  : {len(articles_list):,}")

🔗 Decoding 1500 URLs...



🔗 Decoding:   0%|          | 0/1500 [00:00<?, ?url/s]


   ✅ Decoded : 1,500
   ❌ Failed  : 0
   📄 Unique  : 1,273


In [7]:
def extract_content(url: str) -> tuple[str, str]:
    """
    Ekstrak konten artikel — versi optimized.
    Returns: (content, error_code)
    """
    if _is_google_domain(url):
        return "", "GOOGLE_URL_NOT_DECODED"

    # ── Fetch halaman ──────────────────────────────────────
    resp = _fetch_with_retry(url)
    if resp is None:
        return "", "FETCH_FAILED"
    if isinstance(resp, str):
        return "", resp  # error code

    content_type = resp.headers.get("Content-Type", "")
    if "text/html" not in content_type and "text/" not in content_type:
        return "", "NOT_HTML"

    # ── Parse HTML ─────────────────────────────────────────
    if resp.encoding and "utf" not in resp.encoding.lower():
        resp.encoding = resp.apparent_encoding

    try:
        soup = BeautifulSoup(resp.content, "lxml")
    except Exception:
        soup = BeautifulSoup(resp.content, "html.parser")

    # ── Remove noise (LEBIH KONSERVATIF) ───────────────────
    _remove_noise(soup)

    # Strategy 1-7: Generic strategies
    text = _try_generic_strategies(soup)
    if text and len(text) > 100:
        return _clean(text), ""

    # Strategy 8: FALLBACK — semua <p> yang cukup panjang
    text = _try_all_paragraphs(soup)
    if text and len(text) > 100:
        return _clean(text), ""

    return "", "NO_CONTENT_FOUND"


def _fetch_with_retry(url: str, max_attempts: int = 2):
    """
    Fetch URL dengan retry dan UA rotation untuk 403.
    Returns: Response object, error string, atau None.
    """
    global SESSION

    for attempt in range(max_attempts):
        try:
            resp = SESSION.get(url, timeout=REQUEST_TIMEOUT)

            # HTTP 403: retry dengan UA berbeda
            if resp.status_code == 403 and attempt == 0:
                _rotate_session()
                continue

            # 403 di attempt terakhir: TETAP COBA PARSE BODY
            # (banyak situs kirim konten walaupun status 403)
            if resp.status_code == 403:
                body_size = len(resp.content)
                if body_size > 1000:
                    # Ada konten, coba parse
                    return resp
                return "HTTP_403"

            # Error lain
            if resp.status_code >= 400:
                return f"HTTP_{resp.status_code}"

            return resp

        except requests.exceptions.ConnectionError:
            return "CONNECTION_ERROR"
        except requests.exceptions.Timeout:
            return "TIMEOUT"
        except Exception as e:
            return type(e).__name__

    return "FETCH_FAILED"


def _remove_noise(soup):
    """
    Hapus elemen noise — LEBIH KONSERVATIF dari v3.
    Hanya hapus jika elemen punya sedikit teks paragraf.
    """
    # Hapus tag yang PASTI noise
    for tag in soup.find_all([
        "script", "style", "iframe", "noscript", "svg",
        "video", "audio", "canvas",
    ]):
        tag.decompose()

    # Hapus nav/footer/header HANYA jika bukan container utama
    for tag_name in ["nav", "footer", "aside"]:
        for tag in soup.find_all(tag_name):
            # Jangan hapus jika berisi banyak paragraf (bisa jadi konten)
            p_count = len(tag.find_all("p"))
            if p_count < 3:
                tag.decompose()

    # Hapus elemen noise berdasarkan class/id — HANYA yang jelas-jelas noise
    # (lebih ketat dari v3: harus exact match, bukan partial)
    noise_exact = re.compile(
        r"^(ad|ads|advert|advertisement|sidebar|"
        r"comment|comments|comment-list|"
        r"social-share|share-buttons|"
        r"newsletter|subscribe|subscription|"
        r"popup|modal|overlay|cookie|"
        r"breadcrumb|pagination)$",
        re.I,
    )

    for attr in ["class", "id"]:
        for tag in soup.find_all(attrs={attr: noise_exact}):
            tag.decompose()

    # Hapus "baca juga" / "read more" blocks (tapi hanya jika pendek)
    related_re = re.compile(
        r"(baca.juga|baca-juga|read.more|related|rekomendasi|trending|populer)",
        re.I,
    )
    for tag in soup.find_all(attrs={"class": related_re}):
        if len(tag.get_text(strip=True)) < 500:
            tag.decompose()
    for tag in soup.find_all(attrs={"id": related_re}):
        if len(tag.get_text(strip=True)) < 500:
            tag.decompose()

def _try_generic_strategies(soup) -> str:
    """Coba 7 strategi generic — sama seperti v3 tapi dengan _extract_text_smart."""
    strategies = [
        lambda: soup.find(attrs={"itemprop": "articleBody"}),
        lambda: soup.find("article"),
        lambda: soup.find("div", class_=re.compile(
            r"(detail[_-]?(?:text|konten|content|isi|berita|news|full)|"
            r"read[_-]?(?:content|body|text)|"
            r"article[_-]?(?:body|content|text|detail)|"
            r"post[_-]?(?:body|content|text)|"
            r"entry[_-]?(?:body|content|text)|"
            r"story[_-]?(?:body|content|text)|"
            r"news[_-]?(?:body|content|text|detail)|"
            r"itp_bodycontent|"
            r"content[-_]?detail|"
            r"body[-_]?article|"
            r"text[-_]?article|"
            r"main[-_]?content|"
            r"page[-_]?content|"
            r"indent)", re.I)),
        lambda: soup.find("div", id=re.compile(
            r"(article|content|story|post|detail|read|news|isi)"
            r"[-_]?(body|content|text|area|full)?$", re.I)),
        lambda: soup.find(attrs={"role": "article"}) or soup.find(attrs={"role": "main"}),
        lambda: soup.find("main"),
        lambda: _find_best_content_div(soup),
    ]

    for strategy in strategies:
        try:
            container = strategy()
            if container:
                text = _extract_text_smart(container)
                if text and len(text) > 100:
                    return text
        except Exception:
            continue

    return ""


def _extract_text_smart(container) -> str:
    """
    ⭐ PERBAIKAN UTAMA: Extract text dari container secara SMART.

    v3 hanya ambil <p> tags → gagal jika konten di <div> atau teks langsung.
    v4: coba <p> dulu, jika gagal, pakai get_text() dengan pembersihan.
    """
    # Metode 1: Ambil dari <p> tags (preferred — lebih bersih)
    paragraphs = container.find_all("p")
    texts = [p.get_text(strip=True) for p in paragraphs if len(p.get_text(strip=True)) > 10]

    if texts:
        result = "\n".join(texts)
        if len(result) > 100:
            return result

    # Metode 2: get_text() dari container — untuk situs yang tidak pakai <p>
    # (ini yang memperbaiki unair.ac.id, dll)
    raw = container.get_text(separator="\n", strip=True)
    if not raw:
        return ""

    # Bersihkan: hapus baris pendek (navigation, labels, dll)
    lines = raw.split("\n")
    content_lines = []
    for line in lines:
        line = line.strip()
        # Simpan baris yang cukup panjang (kemungkinan konten)
        if len(line) > 30:
            content_lines.append(line)
        # Atau baris medium yang bukan noise-like
        elif len(line) > 15 and not re.match(r"^(Share|Like|Comment|Follow|Subscribe|Tags?:|Baca|Read|Home|Menu)", line, re.I):
            content_lines.append(line)

    return "\n".join(content_lines)


def _find_best_content_div(soup):
    """Cari div dengan konten teks terbanyak (improved scoring)."""
    best = None
    best_score = 0

    for div in soup.find_all(["div", "section"]):
        # Hitung berdasarkan teks total, bukan hanya <p> count
        text = div.get_text(strip=True)
        text_len = len(text)

        # Bonus untuk adanya <p> tags
        p_count = len([p for p in div.find_all("p", recursive=True)
                       if len(p.get_text(strip=True)) > 30])

        # Penalti untuk terlalu banyak link (navigasi)
        link_count = len(div.find_all("a"))
        link_ratio = link_count / max(text_len / 100, 1)

        # Score: prefer banyak teks, banyak paragraf, sedikit link
        score = text_len * (1 + p_count * 0.5) / (1 + link_ratio * 2)

        # Minimum: harus ada setidaknya 200 char teks
        if text_len > 200 and score > best_score:
            best_score = score
            best = div

    return best


def _try_all_paragraphs(soup) -> str:
    """Last resort: ambil semua <p> di halaman."""
    all_p = soup.find_all("p")
    # Turunkan threshold dari 40 ke 25 karakter
    texts = [p.get_text(strip=True) for p in all_p if len(p.get_text(strip=True)) > 25]
    return "\n".join(texts) if texts else ""


def _clean(text: str) -> str:
    text = re.sub(r"\n{3,}", "\n\n", text)
    text = re.sub(r"[ \t]+", " ", text)
    lines = text.strip().split("\n")
    while lines and len(lines[0].strip()) < 10:
        lines.pop(0)
    while lines and len(lines[-1].strip()) < 10:
        lines.pop()
    return "\n".join(lines).strip()


print("✅ Content extractor ready (smart extraction + site-specific selectors).")

✅ Content extractor ready (smart extraction + site-specific selectors).


# Content Scrapping

In [8]:

def save_checkpoint(articles, path):
    pd.DataFrame(articles).to_csv(path, index=False, encoding="utf-8-sig")

def load_checkpoint(path):
    if os.path.exists(path):
        df = pd.read_csv(path, encoding="utf-8-sig", keep_default_na=False)
        return df.to_dict("records")
    return []


# Resume
existing = load_checkpoint(CHECKPOINT_CSV)
if existing:
    scraped_urls = {_url_hash(str(a.get("link", ""))) for a in existing
                    if a.get("content")}
    remaining = [a for a in articles_list if _url_hash(a["link"]) not in scraped_urls]
    print(f"📂 Checkpoint: {len(existing)} done, {len(remaining)} remaining.")
else:
    existing = []
    remaining = articles_list

# Scrape
if remaining:
    print(f"📝 Scraping {len(remaining)} artikel...\n")

    completed = list(existing)
    ok_count = sum(1 for a in existing if a.get("content") and not a.get("error"))
    fail_count = sum(1 for a in existing if not a.get("content") or a.get("error"))

    pbar = tqdm(total=len(remaining), desc="📝 Scraping", unit="art")

    for i, art in enumerate(remaining):
        content, error = extract_content(art["link"])
        art["content"] = content
        art["error"] = error

        if content:
            ok_count += 1
        else:
            fail_count += 1

        completed.append(art)
        rate = ok_count / max(ok_count + fail_count, 1) * 100
        pbar.set_postfix({"✓": ok_count, "✗": fail_count, "rate": f"{rate:.0f}%"})
        pbar.update(1)

        # Rotate UA setiap 100 request
        if (i + 1) % 100 == 0:
            _rotate_session()

        if len(completed) % CHECKPOINT_EVERY == 0:
            save_checkpoint(completed, CHECKPOINT_CSV)

        _delay()

    pbar.close()
    save_checkpoint(completed, CHECKPOINT_CSV)
    articles_list = completed
else:
    articles_list = existing
    print("✅ All done from checkpoint.")

📝 Scraping 1273 artikel...



📝 Scraping:   0%|          | 0/1273 [00:00<?, ?art/s]

In [9]:
df = pd.DataFrame(articles_list)
for col in ["content", "error"]:
    if col not in df.columns:
        df[col] = ""
    df[col] = df[col].fillna("")

df = df.drop_duplicates(subset=["link"], keep="first").reset_index(drop=True)
df["status"] = df.apply(lambda r: "OK" if r["content"] and not r["error"] else "FAIL", axis=1)
df["content_len"] = df["content"].str.len()

ok = (df["status"] == "OK").sum()
fail = (df["status"] == "FAIL").sum()

print("=" * 60)
print("📊 HASIL SCRAPING v4")
print("=" * 60)
print(f"   Total artikel       : {len(df):,}")
print(f"   ✅ Berhasil (OK)    : {ok:,} ({ok/max(len(df),1)*100:.1f}%)")
print(f"   ❌ Gagal (FAIL)     : {fail:,} ({fail/max(len(df),1)*100:.1f}%)")

if ok > 0:
    ok_df = df[df["status"] == "OK"]
    print(f"   Rentang tanggal     : {df['date'].min()} → {df['date'].max()}")
    print(f"   Rata-rata panjang   : {ok_df['content_len'].mean():,.0f} karakter")
    print(f"   Median panjang      : {ok_df['content_len'].median():,.0f} karakter")

print("=" * 60)

if fail > 0:
    print("\n⚠️ Error breakdown:")
    errs = df[df["status"] == "FAIL"]["error"].value_counts().head(10)
    for err_type, cnt in errs.items():
        print(f"   {err_type or 'UNKNOWN':30s} {cnt:>5,} ({cnt/fail*100:.1f}%)")

df[["date", "title", "status", "content_len"]].head(15)

# %%
df_export = df[df["status"] == "OK"][["date", "link", "title", "content"]].copy()
df_export = df_export.reset_index(drop=True)
df_export.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")

filesize = os.path.getsize(OUTPUT_CSV) / 1024 / 1024
print(f"💾 {OUTPUT_CSV}: {len(df_export):,} baris ({filesize:.2f} MB)")

if os.path.exists(CHECKPOINT_CSV):
    os.remove(CHECKPOINT_CSV)
    print("🗑️ Checkpoint dihapus.")

📊 HASIL SCRAPING v4
   Total artikel       : 1,273
   ✅ Berhasil (OK)    : 1,149 (90.3%)
   ❌ Gagal (FAIL)     : 124 (9.7%)
   Rentang tanggal     : 2025-01-20 08:00 → 2026-03-09 04:51
   Rata-rata panjang   : 4,335 karakter
   Median panjang      : 2,806 karakter

⚠️ Error breakdown:
   HTTP_403                          61 (49.2%)
   NO_CONTENT_FOUND                  61 (49.2%)
   CONNECTION_ERROR                   2 (1.6%)
💾 news_articles_trump_tariff.csv: 1,149 baris (5.00 MB)
🗑️ Checkpoint dihapus.


# Filtering

In [10]:
MUST_HAVE_KEYWORDS = [
    "trump", "amerika serikat", "united states", "AS",
    "washington", "white house", "gedung putih",
]

CONTEXT_KEYWORDS = [
    "tarif", "tariff", "bea masuk", "bea cukai",
    "perang dagang", "trade war", "ekspor", "impor",
    "perdagangan", "proteksionisme", "resiprokal",
    "reciprocal", "sanksi", "trade deal", "negosiasi dagang",
    "supply chain", "investasi", "ekonomi", "industri",
]

BLACKLIST_KEYWORDS = [
    "tarif listrik", "tarif air", "tarif tol",       # tarif bukan dagang
    "tarif ojol", "tarif taksi", "tarif parkir",
    "tarif rumah sakit", "tarif hotel",
    "donald trump jr", "trump hotel", "trump golf",   # Trump non-politik
    "ivanka", "melania",
    "judi", "judol", "slot",
    "drakor", "film", "anime", "sinetron",
    "resep", "kuliner", "wisata", "olahraga",
    "gempa", "banjir", "cuaca",
]

def is_relevant(row):
    text = f"{row['title']} {row['content']}".lower()

    # Step 1: Blacklist
    if any(b in text for b in BLACKLIST_KEYWORDS):
        return False

    # Step 2: Harus ada aktor utama (Trump / AS)
    if not any(k in text for k in MUST_HAVE_KEYWORDS):
        return False

    # Step 3: Harus ada konteks perdagangan
    if not any(k in text for k in CONTEXT_KEYWORDS):
        return False

    return True

# Pola konten sampah (bukan artikel asli)
JUNK_PATTERNS = [
    "checking if the site connection is secure",
    "needs to review the security of your connection",
    "please enable javascript",
    "enable cookies to continue",
    "just a moment",
    "attention required",
    "access denied",
    "are you a human",
    "complete the security check",
    "ray id",
    "cloudflare",
    "please verify you are a human",
    "browser is out of date",
    "please turn javascript on",
    "this process is automatic",
    "redirecting",
    "your browser will redirect",
    "security check by",
    "ddos protection by",
]

def is_junk(row):
    content = row["content"].lower().strip()
    # Terlalu pendek → kemungkinan bukan artikel
    if len(content) < 150:
        return True
    # Cocok dengan pola halaman security/challenge
    if any(p in content for p in JUNK_PATTERNS):
        return True
    return False

before_count = len(df_export)

# Step 1: Buang artikel tidak relevan
df_export = df_export[df_export.apply(is_relevant, axis=1)]

after_relevance = len(df_export)

# Step 2: Buang konten sampah (Cloudflare, JS challenge, dll)
df_export = df_export[~df_export.apply(is_junk, axis=1)]

df_export = df_export.reset_index(drop=True)
after_junk = len(df_export)

# Re-save CSV
df_export.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")
filesize = os.path.getsize(OUTPUT_CSV) / 1024 / 1024

print(f"🔍 Filter results:")
print(f"   Total OK           : {before_count:,}")
print(f"   Tidak relevan      : -{before_count - after_relevance:,}")
print(f"   Konten sampah      : -{after_relevance - after_junk:,}")
print(f"   ✅ Final           : {after_junk:,} artikel")
print(f"💾 {OUTPUT_CSV} ({filesize:.2f} MB)")

🔍 Filter results:
   Total OK           : 1,149
   Tidak relevan      : -178
   Konten sampah      : -13
   ✅ Final           : 958 artikel
💾 news_articles_trump_tariff.csv (3.99 MB)


In [11]:
df_ok = df[df["status"] == "OK"].copy()

if not df_ok.empty:
    print("📊 Distribusi panjang konten:\n")
    bins = [0, 500, 1000, 2000, 5000, 10000, float("inf")]
    labels = ["<500", "500-1K", "1K-2K", "2K-5K", "5K-10K", ">10K"]
    df_ok["len_bin"] = pd.cut(df_ok["content_len"], bins=bins, labels=labels)
    dist = df_ok["len_bin"].value_counts().sort_index()
    for label, count in dist.items():
        bar = "█" * int(count / max(dist.max(), 1) * 30)
        print(f"   {label:>8s} │ {bar} {count:,}")

    df_ok["domain"] = df_ok["link"].apply(lambda u: urlparse(u).netloc)
    print(f"\n📊 Top 15 sumber:")
    for domain, count in df_ok["domain"].value_counts().head(15).items():
        print(f"   {domain:45s} {count:>4d}")

    sample = df_ok.sample(1).iloc[0]
    print(f"\n{'='*60}")
    print(f"📰 {sample['title'][:70]}")
    print(f"📅 {sample['date']} | 🔗 {urlparse(sample['link']).netloc}")
    print(f"{'='*60}")
    print(sample["content"][:600])
    print("...")

📊 Distribusi panjang konten:

       <500 │ ██ 42
     500-1K │ ████ 80
      1K-2K │ ████████████ 219
      2K-5K │ ██████████████████████████████ 516
     5K-10K │ █████████ 170
       >10K │ ███████ 122

📊 Top 15 sumber:
   www.cnbcindonesia.com                          219
   www.bloombergtechnoz.com                       118
   www.kompas.id                                  107
   www.metrotvnews.com                             64
   www.cnnindonesia.com                            61
   www.tempo.co                                    61
   news.detik.com                                  44
   money.kompas.com                                39
   finance.detik.com                               29
   www.dw.com                                      25
   www.antaranews.com                              24
   kumparan.com                                    24
   ekbis.sindonews.com                             18
   ekonomi.bisnis.com                              16
   www.bbc.com      

In [12]:
def diagnose_url(url):
    print(f"🔍 {url[:90]}")
    domain = urlparse(url).netloc
    print(f"   Domain: {domain} | Google: {_is_google_domain(url)}")

    if _is_google_domain(url):
        print("   ⚠️ Masih Google URL — decode gagal!")
        return

    try:
        resp = SESSION.get(url, timeout=REQUEST_TIMEOUT)
        ct = resp.headers.get("Content-Type", "N/A")
        print(f"   HTTP {resp.status_code} | {ct} | {len(resp.content):,} bytes")

        if "text/html" not in ct:
            print(f"   ⚠️ Bukan HTML!")
            return

        soup = BeautifulSoup(resp.content, "lxml")

        # Check containers
        for name, finder in [
            ("itemprop=articleBody", lambda: soup.find(attrs={"itemprop": "articleBody"})),
            ("<article>", lambda: soup.find("article")),
            ("<main>", lambda: soup.find("main")),
            ("role=main", lambda: soup.find(attrs={"role": "main"})),
        ]:
            found = finder()
            if found:
                p_tags = len(found.find_all("p"))
                text_len = len(found.get_text(strip=True))
                print(f"   ✅ {name}: {p_tags} <p>, {text_len:,} chars text")
            else:
                print(f"   ❌ {name}")

        all_p = soup.find_all("p")
        long_p = [p for p in all_p if len(p.get_text(strip=True)) > 25]
        print(f"   Total <p>: {len(all_p)} | Useful (>25ch): {len(long_p)}")

        body_text = soup.get_text(strip=True)
        if len(body_text) < 200:
            print("   ⚠️ Sangat sedikit teks — butuh JavaScript!")

    except Exception as e:
        print(f"   ❌ {e}")


# Run
fail_df = df[df["status"] == "FAIL"].head(5)
if not fail_df.empty:
    print("🔍 DIAGNOSTIK:\n")
    for _, row in fail_df.iterrows():
        diagnose_url(row["link"])
        print(f"   Error: {row['error']}\n")
else:
    print("🎉 Semua berhasil!")

🔍 DIAGNOSTIK:

🔍 https://jakartaglobe.id/business/rupiah-slumps-as-uschina-trade-war-escalates
   Domain: jakartaglobe.id | Google: False
   HTTP 403 | text/html | 919 bytes
   ❌ itemprop=articleBody
   ❌ <article>
   ❌ <main>
   ❌ role=main
   Total <p>: 0 | Useful (>25ch): 0
   Error: HTTP_403

🔍 https://chatnews.id/read/harga-bitcoin-anjlok-investor-khawatir-tarif-impor-as-terus-tekan
   Domain: chatnews.id | Google: False
   HTTP 200 | text/html;charset=utf-8 | 78,822 bytes
   ❌ itemprop=articleBody
   ❌ <article>
   ✅ <main>: 0 <p>, 33 chars text
   ❌ role=main
   Total <p>: 0 | Useful (>25ch): 0
   ⚠️ Sangat sedikit teks — butuh JavaScript!
   Error: NO_CONTENT_FOUND

🔍 https://rri.co.id/bandar-lampung/sudut-pandang/1441687/merespons-trump-tarif-saatnya-indon
   Domain: rri.co.id | Google: False
   HTTP 403 | text/html; charset=UTF-8 | 6,133 bytes
   ❌ itemprop=articleBody
   ❌ <article>
   ❌ <main>
   ✅ role=main: 0 <p>, 41 chars text
   Total <p>: 0 | Useful (>25ch): 0
   ⚠️ Sa

# Rescrape Failed Article

In [13]:
def rescrape_failed(df, max_retry=None):
    """Re-scrape artikel yang gagal."""
    fail_df = df[df["status"] == "FAIL"].copy()

    # Skip yang pasti gagal (Google URL)
    retryable = fail_df[fail_df["error"] != "GOOGLE_URL_NOT_DECODED"]

    if max_retry:
        retryable = retryable.head(max_retry)

    if retryable.empty:
        print("Tidak ada artikel yang bisa di-retry.")
        return df

    print(f"🔄 Retrying {len(retryable)} artikel...\n")
    _rotate_session()  # Fresh UA

    recovered = 0
    pbar = tqdm(total=len(retryable), desc="🔄 Retry", unit="art")

    for idx, row in retryable.iterrows():
        content, error = extract_content(row["link"])

        if content:
            df.at[idx, "content"] = content
            df.at[idx, "error"] = ""
            df.at[idx, "status"] = "OK"
            df.at[idx, "content_len"] = len(content)
            recovered += 1

        pbar.set_postfix({"recovered": recovered})
        pbar.update(1)
        _delay()

    pbar.close()

    ok = (df["status"] == "OK").sum()
    total = len(df)
    print(f"\n✅ Recovered: {recovered} | New total OK: {ok}/{total} ({ok/total*100:.1f}%)")

    # Re-export
    df_export = df[df["status"] == "OK"][["date", "link", "title", "content"]]
    df_export.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")
    print(f"💾 Updated: {OUTPUT_CSV}")

    return df

In [14]:
# rescrape yang failed
df = rescrape_failed(df)

🔄 Retrying 124 artikel...



🔄 Retry:   0%|          | 0/124 [00:00<?, ?art/s]


✅ Recovered: 2 | New total OK: 1151/1273 (90.4%)
💾 Updated: news_articles_trump_tariff.csv


# Quality Checks

In [38]:
df_check = pd.read_csv('news_articles_trump_tariff.csv')
df_check

,date,link,title,content
0,2025-01-21 08:00,https://www.cnbcindonesia.com/research/2025012...,Trump Sebar Exceutive Order: Emang Semengerika...,"Jakarta, CNBC Indonesia -Amerika Serikat (AS) ..."
1,2025-01-21 08:00,https://www.cnbcindonesia.com/research/2025012...,"Alasan Rupiah 'Berpesta' di Pelantikan Trump, ...","Jakarta, CNBC Indonesia -Nilai tukar rupiah te..."
2,2025-01-20 08:00,https://portalhukum.id/opini-hukum/lambannya-p...,Lambannya Penegakan Hukum di Kampar dalam Kasu...,"Dalam beberapa tahun terakhir, kasus korupsi d..."
3,2025-01-22 08:00,https://www.cnbcindonesia.com/research/2025012...,"Trump Beri Kabar Baik, Saatnya Menunggu Dolar ...","Jakarta, CNBC Indonesia-Pasar keuangan Indones..."
4,2025-03-04 08:00,https://www.cnbcindonesia.com/research/2025030...,"IHSG Merah Lagi, Begini Penjelasan dari Analis...","Jakarta, CNBC Indonesia -Indeks Harga Saham Ga..."
...,...,...,...,...
1146,2025-06-21 07:00,https://www.liputan6.com/bisnis/read/6058462/d...,Donald Trump Kembali Tekan Ketua The Fed Jerom...,Presiden Amerika Serikat (AS) Donald Trump men...
1147,2025-06-22 07:00,https://kabar24.bisnis.com/read/20250622/19/18...,"Donald Trump Serang Iran, Ekonomi Global Bakal...","Bisnis.com,JAKARTA — Serangan Amerika Serikat ..."
1148,2025-06-20 07:00,https://www.cnbcindonesia.com/tech/20250620153...,Trump 'Palak' Raksasa Teknologi Rp 900 Triliun...,"Jakarta, CNBC Indonesia -Perusahaan semikonduk..."
1149,2025-06-27 07:00,https://www.dw.com/id/vietnam-bersiap-hadapi-b...,Vietnam Bersiap Hadapi Berakhirnya Keringanan ...,Penangguhan tarif 46% dari Amerika Serikat ter...


In [39]:
# ══════════════════════════════════════════════════════════════
#  RELEVANCE + QUALITY FILTER
# ══════════════════════════════════════════════════════════════

MUST_HAVE_KEYWORDS = [
    "trump", "amerika serikat", "united states", "AS",
    "washington", "white house", "gedung putih",
]

CONTEXT_KEYWORDS = [
    "tarif", "tariff", "bea masuk", "bea cukai",
    "perang dagang", "trade war", "ekspor", "impor",
    "perdagangan", "proteksionisme", "resiprokal",
    "reciprocal", "sanksi", "trade deal", "negosiasi dagang",
    "supply chain", "investasi", "ekonomi", "industri",
]

BLACKLIST_KEYWORDS = [
    "tarif listrik", "tarif air", "tarif tol",
    "tarif ojol", "tarif taksi", "tarif parkir",
    "tarif rumah sakit", "tarif hotel",
    "donald trump jr", "trump hotel", "trump golf",
    "ivanka", "melania",
    "judi", "judol", "slot",
    "drakor", "film", "anime", "sinetron",
    "resep", "kuliner", "wisata", "olahraga",
    "gempa", "cuaca",
]

JUNK_PATTERNS = [
    "checking if the site connection is secure",
    "needs to review the security of your connection",
    "please enable javascript",
    "enable cookies to continue",
    "just a moment",
    "attention required",
    "access denied",
    "are you a human",
    "complete the security check",
    "ray id",
    "cloudflare",
    "please verify you are a human",
    "browser is out of date",
    "please turn javascript on",
    "this process is automatic",
    "redirecting",
    "your browser will redirect",
    "security check by",
    "ddos protection by",
]

# ── Fungsi filter dengan alasan ──────────────────────────────
def get_reject_reason(row):
    """Return alasan penolakan, atau None jika artikel OK."""
    text = f"{row['title']} {row['content']}".lower()
    content = row["content"].lower().strip()

    # Junk check
    if len(content) < 150:
        return "junk: konten terlalu pendek"
    if any(p in content for p in JUNK_PATTERNS):
        return "junk: halaman security/cloudflare"

    # Blacklist check
    matched_blacklist = [b for b in BLACKLIST_KEYWORDS if b in text]
    if matched_blacklist:
        return f"blacklist: {', '.join(matched_blacklist[:3])}"

    # Must-have check
    if not any(k in text for k in MUST_HAVE_KEYWORDS):
        return "tidak relevan: tidak ada aktor utama (Trump/AS)"

    # Context check
    if not any(k in text for k in CONTEXT_KEYWORDS):
        return "tidak relevan: tidak ada konteks perdagangan"

    return None  # ✅ artikel lolos

# ── Terapkan filter & pisahkan ───────────────────────────────
before_count = len(df_check)

df_check["_reject_reason"] = df_check.apply(get_reject_reason, axis=1)

df_relevant   = df_check[df_check["_reject_reason"].isna()].drop(columns=["_reject_reason"]).reset_index(drop=True)
df_irrelevant = df_check[df_check["_reject_reason"].notna()].rename(columns={"_reject_reason": "reject_reason"}).reset_index(drop=True)

after_count = len(df_relevant)

# ── Statistik ────────────────────────────────────────────────
print(f"🔍 Filter results:")
print(f"   Total masuk        : {before_count:,}")
print(f"   ✅ Relevan         : {after_count:,} artikel")
print(f"   ❌ Tidak relevan   : {before_count - after_count:,} artikel")
print()

# Breakdown alasan penolakan
print("📊 Breakdown alasan penolakan:")
print(df_irrelevant["reject_reason"].str.split(":").str[0].value_counts().to_string())
print()

# Preview artikel tidak relevan
print("👀 Sample artikel tidak relevan:")
df_irrelevant.to_csv("irrelevant_articles.csv", index=False, encoding="utf-8-sig")

🔍 Filter results:
   Total masuk        : 1,151
   ✅ Relevan         : 1,020 artikel
   ❌ Tidak relevan   : 131 artikel

📊 Breakdown alasan penolakan:
reject_reason
blacklist        72
tidak relevan    41
junk             18

👀 Sample artikel tidak relevan:


In [45]:
df_relevant.drop_duplicates()
df_relevant.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1020 entries, 0 to 1019
Data columns (total 4 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   date     1020 non-null   object
 1   link     1020 non-null   object
 2   title    1020 non-null   object
 3   content  1020 non-null   object
dtypes: object(4)
memory usage: 32.0+ KB


In [46]:
df_relevant = df_relevant[~df_relevant["content"].str.contains(
    "Checking if the site connection is secure",
    case=False, na=False
)].reset_index(drop=True)

print(f"✅ Remaining: {len(df_relevant):,} artikel")

✅ Remaining: 1,020 artikel


In [47]:
df_relevant.to_csv('articles.csv', index=False, encoding="utf-8-sig")

In [48]:
df_relevant

,date,link,title,content
0,2025-01-21 08:00,https://www.cnbcindonesia.com/research/2025012...,Trump Sebar Exceutive Order: Emang Semengerika...,"Jakarta, CNBC Indonesia -Amerika Serikat (AS) ..."
1,2025-01-21 08:00,https://www.cnbcindonesia.com/research/2025012...,"Alasan Rupiah 'Berpesta' di Pelantikan Trump, ...","Jakarta, CNBC Indonesia -Nilai tukar rupiah te..."
2,2025-01-22 08:00,https://www.cnbcindonesia.com/research/2025012...,"Trump Beri Kabar Baik, Saatnya Menunggu Dolar ...","Jakarta, CNBC Indonesia-Pasar keuangan Indones..."
3,2025-03-04 08:00,https://www.cnbcindonesia.com/research/2025030...,"IHSG Merah Lagi, Begini Penjelasan dari Analis...","Jakarta, CNBC Indonesia -Indeks Harga Saham Ga..."
4,2025-03-19 07:00,https://indodax.com/academy/bitcoin-200k-predi...,Bernstein: Bitcoin Bisa Naik 2x Lipat! Target ...,HargaBitcoin(BTC)pernah melewati angka terting...
...,...,...,...,...
1015,2025-06-19 07:00,https://voi.id/teknologi/489452/produsen-mesin...,Produsen Mesin Bitcoin China Pindahkan Produks...,JAKARTA - Tiga produsen mesin penambang bitcoi...
1016,2025-06-21 07:00,https://www.liputan6.com/bisnis/read/6058462/d...,Donald Trump Kembali Tekan Ketua The Fed Jerom...,Presiden Amerika Serikat (AS) Donald Trump men...
1017,2025-06-22 07:00,https://kabar24.bisnis.com/read/20250622/19/18...,"Donald Trump Serang Iran, Ekonomi Global Bakal...","Bisnis.com,JAKARTA — Serangan Amerika Serikat ..."
1018,2025-06-20 07:00,https://www.cnbcindonesia.com/tech/20250620153...,Trump 'Palak' Raksasa Teknologi Rp 900 Triliun...,"Jakarta, CNBC Indonesia -Perusahaan semikonduk..."
